In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.api.types import is_numeric_dtype
import plotly.express as px
import seaborn as sns
import math
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

# 1. Load and inspect the data

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/salioudunn11/Fitness_Prediction/main/gym_members_exercise_tracking.csv")

In [ ]:
display(df.head())

In [ ]:
display(df.shape)

In [ ]:
display(df.dtypes)

# 2. Summary Statistics

In [ ]:
display(df.describe())

In [ ]:
# it seems redundant but we wanted to see this four vlaues in clear format
display(df.describe().T[['mean', 'std', 'min', 'max']].round(2))

In [ ]:
display(df['Workout_Type'].value_counts())

In [ ]:
display(df['Gender'].value_counts())

# 3. Data Visualization

### 3.1 Calories Burned Distribution

In [ ]:
df['Calories_Burned'].hist(bins=10)
plt.title("Distribution of Calories Burned Per Session")
plt.xlabel("Calories")
plt.ylabel("Frequency")
plt.show()

### 3.2 Numerical Attribute Boxplots

In [ ]:
numeric_cols = [
    'Age', 'Weight (kg)', 'Height (m)', 'Max_BPM',
    'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)',
    'Calories_Burned', 'Fat_Percentage', 'Water_Intake (liters)',
    'Workout_Frequency (days/week)','Experience_Level', 'BMI'
]
# Keep only columns actually present in df
plot_cols = [c for c in numeric_cols if c in df.columns]

# 2. Setup grid dimensions
n_cols = 3
n_rows = math.ceil(len(plot_cols) / n_cols)

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(14, n_rows * 3.2))
axes = axes.flatten()

# Custom style for outlier markers
flierprops = dict(marker='o', markersize=5, markerfacecolor='#e74c3c', markeredgecolor='none', alpha=0.7)

for i, col in enumerate(plot_cols):
    ax = axes[i]

    # width=0.35 keeps boxes sleek rather than filling the entire card
    sns.boxplot(
        y=df[col],
        ax=ax,
        width=0.35,
        color='#4a90e2',
        linewidth=1.2,
        flierprops=flierprops
    )

    # Clean styling
    ax.set_title(col, fontsize=11, fontweight='semibold', pad=10)
    ax.set_ylabel("")
    ax.grid(axis='y', linestyle=':', alpha=0.6)

# Remove unused axes in the grid
for j in range(len(plot_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Outlier Analysis across Numerical Attributes", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.3 Correlation Matrix

In [ ]:
plt.figure(figsize=(12, 10))
numeric_cols = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_cols.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Color-Coded Pearson Correlation Matrix')
plt.show()

### 3.4 Calories by Workout Type & Gender

In [ ]:
print("Correlation of each feature with 'Calories_Burned':")
display(correlation_matrix['Calories_Burned'].round(2).sort_values(ascending=False))

###3.5 Categorical vs. Target Analysis

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x='Workout_Type', y='Calories_Burned', hue='Gender', errorbar=None, palette='muted')
plt.title("Mean Calories Burned Across Workout Types by Gender")
plt.ylabel("Average Calories Burned")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig = px.scatter_matrix(
    df,
    dimensions=['Session_Duration (hours)', 'Calories_Burned', 'Avg_BPM', 'Fat_Percentage'],
    color='Workout_Type',
    title="Scatter Matrix of Gym Attributes by Workout Type",
    width=900,
    height=900
)
fig.show()

# 4. Data Cleaning and Preprocessing

### 4.1 Missing Values & Duplicates

In [ ]:
display(df.isna().sum())

In [ ]:
display(df.duplicated().sum())

### 4.2 Unit Conversions

In [ ]:
# 1. Create an independent working copy to preserve raw data
df_clean = df.copy()

# 2. Add imperial conversions
df_clean['Weight (lb)'] = (df_clean['Weight (kg)'] * 2.20462).round(2)
df_clean['Height (in)'] = (df_clean['Height (m)'] * 39.3701).round(2)

# 3. Drop the metric columns
df_clean = df_clean.drop(columns=['Weight (kg)', 'Height (m)'])

# 4. Inspect result
display(df_clean.head())

### 4.3 Z-Score Outlier Detection

In [ ]:
num_data = df_clean.select_dtypes(include=[np.number])
Z = (num_data - num_data.mean()) / num_data.std()

def highlight_outliers(val):
    if abs(val) > 3:
        return 'background-color: #ff4d4d; color: white; font-weight: bold;'
    return 'color: #555555;'

# Filter rows that actually contain at least one outlier to keep it concise
outlier_rows = Z[(Z.abs() > 3).any(axis=1)]

styled_outliers = (
    outlier_rows.style
    .map(highlight_outliers)
    .format("{:+.2f}")
    .set_caption("🚩 Records with Extreme Outliers (|Z| > 3.0)")
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '14px'), ('font-weight', 'bold'), ('margin-bottom', '8px')]}
    ])
)

display(styled_outliers)

In [ ]:
outlier_mask = (Z.abs() > 3).any(axis=1)
outliers = df_clean.loc[outlier_mask]
print(f'Number of Total Outlier: {len(outliers)}')

# 5. Discretization

### 5.1 Equal-Width Binning

In [ ]:
cal_bins_width = pd.cut(df_clean['Calories_Burned'], 4)
print("Equal-width bins for Calories_Burned:")
display(cal_bins_width.value_counts(sort=False))

In [ ]:
fat_bins_width = pd.cut(df_clean['Fat_Percentage'], 4)
print("Equal-width bins for Fat_Percentage:")
display(fat_bins_width.value_counts(sort=False))

### 5.2 Equal-Frequency Binning

In [ ]:
cal_bins_freq = pd.qcut(df_clean['Calories_Burned'], 4)
print("Equal-frequency bins for Calories_Burned:")
display(cal_bins_freq.value_counts(sort=False))

In [ ]:
fat_bins_freq = pd.qcut(df_clean['Fat_Percentage'], 4)
print("Equal-frequency bins for Fat_Percentage:")
display(fat_bins_freq.value_counts(sort=False))

### 5. MULTIVARIABLE INTERACTIONS
Scatter Matrix: Duration, Calories, Heart Rate & Fat Percentage

* LINEAR CONVERGENCE:
  - `Session_Duration` vs. `Calories_Burned`: Forms an exceptionally tight diagonal band confirming strong linear separability for regression.

* CLUSTER SEGREGATION:
  - `Avg_BPM` vs. `Workout_Type`: Displays vertical cluster bands; HIIT points concentrate between 145–169 BPM, while Yoga points populate the lower 120–135 BPM range.

* INVERSE STEP PHENOMENON:
  - `Fat_Percentage` vs. `Session_Duration`: Distinct grouping where members with body fat $<20\%$ frequently maintain 1.5–2.0 hour sessions, while $>25\%$ fat clusters in shorter durations.

# 6. Modeling

## 6.1 Entropy, decision tree, overfitting and measurement using categorical data

In [ ]:
def calculate_entropy(series):
    probs = series.value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs + 1e-9)) # 1e-9 prevents log2(0)

def calculate_info_gain(df, split_col, target_col):
    parent_entropy = calculate_entropy(df[target_col])
    total_len = len(df)
    weighted_child_entropy = 0
    for val, subset in df.groupby(split_col):
        weight = len(subset) / total_len
        weighted_child_entropy += weight * calculate_entropy(subset[target_col])
    return parent_entropy - weighted_child_entropy
target = 'Workout_Type'
print(f"Parent Entropy ({target}): {calculate_entropy(df_clean[target]):.4f}")
for feature in ['Gender', 'Experience_Level']:
    gain = calculate_info_gain(df_clean, feature, target)
    print(f"Information Gain splitting on {feature}: {gain:.4f}")

In [ ]:
# --- 7. DECISION TREE CLASSIFICATION---
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Prepare features (X) and target (y)
# Encode categorical 'Gender' into numeric
X = pd.get_dummies(df_clean.drop(columns=['Workout_Type']), drop_first=True)
y = df_clean['Workout_Type']

# Holdout method: 80% Training, 20% Testing (Lecture 9, Slide 21)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Build an initial tree with max_depth=3 for interpretability
dt_base = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
dt_base.fit(X_train, y_train)

# Plot the Decision Tree structure
plt.figure(figsize=(18, 10))
plot_tree(dt_base, feature_names=X.columns, class_names=dt_base.classes_, filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree Classifier (Entropy Criterion, Max Depth = 3)")
plt.show()

In [ ]:
# --- 8. OVERFITTING & MODEL SELECTION ---
from sklearn.model_selection import cross_val_score

depths = range(1, 16)
train_errors = []
test_errors = []

for d in depths:
    clf = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_errors.append(1 - clf.score(X_train, y_train))
    test_errors.append(1 - clf.score(X_test, y_test))

# Plot Training vs. Test Error
plt.figure(figsize=(10, 6))
plt.plot(depths, train_errors, marker='o', label='Training Error', color='blue')
plt.plot(depths, test_errors, marker='s', label='Test Error', color='red')
plt.xlabel('Tree Max Depth (Model Complexity)')
plt.ylabel('Classification Error Rate (1 - Accuracy)')
plt.title('Underfitting vs. Overfitting Curve')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Select optimal depth (lowest test error) and run 5-Fold Stratified Cross-Validation
optimal_depth = depths[np.argmin(test_errors)]
print(f"Optimal Max Depth: {optimal_depth}")

best_dt = DecisionTreeClassifier(criterion='entropy', max_depth=optimal_depth, random_state=42)
cv_scores = cross_val_score(best_dt, X, y, cv=5) # Stratified by default for classifiers
print(f"5-Fold Cross-Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
# --- 9. MEASURING PERFORMANCE ---
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize

best_dt.fit(X_train, y_train)
y_pred = best_dt.predict(X_test)

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=best_dt.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_dt.classes_)
disp.plot(cmap='Blues')
plt.title(f"Confusion Matrix (Pruned Tree, Depth={optimal_depth})")
plt.show()

# 2. Precision, Recall, F1-Score
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

# 3. Multi-class ROC Curve (One-vs-Rest) & AUC
y_test_bin = label_binarize(y_test, classes=best_dt.classes_)
y_score = best_dt.predict_proba(X_test)

plt.figure(figsize=(8, 6))
for i, class_name in enumerate(best_dt.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.50)')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR / Recall)')
plt.title('Multi-Class ROC Curves (One-vs-Rest)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.5)
plt.show()

I don't see any meaningful from using categorical, the result is extremely overfitting and ROC curve show the result doesn't give any better than random guess.

## 6.2 Modeling

In [ ]:
# --- 6. BEST SPLIT SELECTION & DECREASING ENTROPY TRACKER ---
# Target: Binary High_Calorie_Burn (Median split -> Parent Entropy ≈ 1.0)
median_cal = df_clean['Calories_Burned'].median()
df_tree = df_clean.copy()
df_tree['High_Calorie_Burn'] = np.where(df_tree['Calories_Burned'] > median_cal, 'Yes', 'No')

# Candidate predictors (excluding target & raw Calories_Burned to prevent leakage)
candidate_features = [
    'Session_Duration (hours)', 'Avg_BPM', 'Max_BPM', 'Resting_BPM',
    'Fat_Percentage', 'Age', 'Weight (lb)', 'Height (in)', 'BMI',
    'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'Experience_Level',
    'Gender', 'Workout_Type'
]

def get_entropy(y_series):
    probs = y_series.value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs + 1e-12))

def find_best_split(data, features, target='High_Calorie_Burn'):
    parent_ent = get_entropy(data[target])
    n_total = len(data)
    results = []

    for col in features:
        if pd.api.types.is_numeric_dtype(data[col]):
            # Continuous attribute: scan sorted midpoints
            unique_vals = np.sort(data[col].unique())
            if len(unique_vals) <= 1:
                continue
            midpoints = (unique_vals[:-1] + unique_vals[1:]) / 2
            best_col_ent = float('inf')
            best_thresh = None

            for thresh in midpoints:
                left = data[data[col] <= thresh][target]
                right = data[data[col] > thresh][target]
                w_ent = (len(left)/n_total)*get_entropy(left) + (len(right)/n_total)*get_entropy(right)
                if w_ent < best_col_ent:
                    best_col_ent = w_ent
                    best_thresh = thresh

            results.append({
                'Feature': col,
                'Best_Split_Condition': f"<= {best_thresh:.3f}",
                'Weighted_Child_Entropy': round(best_col_ent, 4),
                'Information_Gain': round(parent_ent - best_col_ent, 4)
            })
        else:
            # Categorical attribute: multi-way split
            w_ent = sum((len(sub)/n_total)*get_entropy(sub[target]) for _, sub in data.groupby(col))
            results.append({
                'Feature': col,
                'Best_Split_Condition': "Multi-way categories",
                'Weighted_Child_Entropy': round(w_ent, 4),
                'Information_Gain': round(parent_ent - w_ent, 4)
            })

    return pd.DataFrame(results).sort_values(by='Information_Gain', ascending=False)

# --- STEP 1: ROOT NODE (LEVEL 0 -> LEVEL 1) ---
root_entropy = get_entropy(df_tree['High_Calorie_Burn'])
print(f"LEVEL 0 (Before Split) — Parent Entropy: {root_entropy:.4f}\n")

level1_ranking = find_best_split(df_tree, candidate_features)
print("Candidate Splits Ranked by Best Information Gain (Lowest Child Entropy):")
display(level1_ranking.head(5))

# Extract the winning Root Split (Session_Duration)
best_root_feat = level1_ranking.iloc[0]['Feature']
best_root_thresh = float(level1_ranking.iloc[0]['Best_Split_Condition'].replace('<= ', ''))
level1_entropy = level1_ranking.iloc[0]['Weighted_Child_Entropy']

# --- STEP 2: LEVEL 2 RECALCULATION (KEEPING ALL FEATURES FOR CONTINUOUS RE-SPLITTING) ---
left_branch = df_tree[df_tree[best_root_feat] <= best_root_thresh]
right_branch = df_tree[df_tree[best_root_feat] > best_root_thresh]

print(f"\n--- LEVEL 2: Left Branch ({best_root_feat} <= {best_root_thresh:.3f}, n={len(left_branch)}) ---")
print(f"Branch Entropy before Level 2 split: {get_entropy(left_branch['High_Calorie_Burn']):.4f}")
left_lvl2 = find_best_split(left_branch, candidate_features)
display(left_lvl2.head(3))

print(f"\n--- LEVEL 2: Right Branch ({best_root_feat} > {best_root_thresh:.3f}, n={len(right_branch)}) ---")
print(f"Branch Entropy before Level 2 split: {get_entropy(right_branch['High_Calorie_Burn']):.4f}")
right_lvl2 = find_best_split(right_branch, candidate_features)
display(right_lvl2.head(3))

# Overall Weighted Entropy at Level 2
level2_weighted_entropy = (
    (len(left_branch) / len(df_tree)) * left_lvl2.iloc[0]['Weighted_Child_Entropy'] +
    (len(right_branch) / len(df_tree)) * right_lvl2.iloc[0]['Weighted_Child_Entropy']
)

print("\n=== ENTROPY PROGRESSION SUMMARY ===")
print(f"Level 0 (Root Node):     {root_entropy:.4f} bits")
print(f"Level 1 (After Split 1): {level1_entropy:.4f} bits  (Drop: -{root_entropy - level1_entropy:.4f})")
print(f"Level 2 (After Split 2): {level2_weighted_entropy:.4f} bits  (Drop: -{level1_entropy - level2_weighted_entropy:.4f})")

This code implements Hunt’s Decision Tree induction algorithm from scratch to demonstrate how a classifier finds the best split and systematically reduces disorder (entropy) across tree levels. First, it converts Calories_Burned into a balanced binary target (High_Calorie_Burn: 'Yes' vs. 'No') with an initial root entropy near 1.0 bit, while excluding raw Calories_Burned from the predictors to prevent data leakage. Next, the find_best_split function evaluates every candidate predictor—scanning all sorted midpoints ($A \le v$) for continuous features and grouping categories for nominal features—to rank splits by Information Gain ($\text{Parent Entropy} - \text{Weighted Child Entropy}$). Finally, it selects the top feature (Session_Duration (hours)) as the Level 1 root node, partitions the dataset into left and right subsets, and recalculates the best Level 2 splits across all candidate features—printing the weighted entropy from Level 0 down to Level 2 to prove that uncertainty continuously decreases at every step

In [ ]:
# --- 7. DECISION TREE VISUALIZATION (SHOWING ENTROPY DROP AT EVERY NODE) ---
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

# 1. Encode categorical predictors (Gender, Workout_Type) into numeric 0/1 columns
X = pd.get_dummies(df_tree[candidate_features], drop_first=True)
y = df_tree['High_Calorie_Burn']

# 2. 80/20 Stratified Holdout Split (used across Sections 8-16 for fair model evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Fit Decision Tree using Entropy Criterion (allows continuous re-splitting at deeper levels)
dt_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
dt_entropy.fit(X_train, y_train)

# 4. Plot the Decision Tree structure
plt.figure(figsize=(22, 11))
plot_tree(
    dt_entropy,
    feature_names=X.columns,
    class_names=['No (Low Cal)', 'Yes (High Cal)'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Split Selection: Continuous Re-Splitting & Decreasing Entropy")
plt.show()

This code builds and visualizes a scikit-learn Decision Tree classifier to automate the entropy-based splitting logic on unseen training data. First, it converts categorical predictors (Gender and Workout_Type) into binary dummy variables using pd.get_dummies and partitions the dataset into an 80% training set and a 20% holdout test set via stratified sampling so both subsets preserve the exact Yes/No class balance. Next, it fits a DecisionTreeClassifier configured with criterion='entropy' and pre-pruned to max_depth=3 for interpretability, allowing continuous features to be re-split at narrower thresholds across deeper levels. Finally, plot_tree renders the resulting hierarchical decision structure—color-coded by class purity—visually confirming how entropy continuously decreases from the root node down to the leaf nodes.

In [ ]:
# --- 8. OVERFITTING, ENTROPY DECAY & PRUNING ---
from sklearn.model_selection import cross_val_score

depths = range(1, 13)
train_errors = []
test_errors = []
weighted_leaf_entropies = []

def get_tree_weighted_entropy(tree_model):
    """Computes the sample-weighted average entropy across all leaf nodes."""
    tree = tree_model.tree_
    is_leaf = (tree.children_left == -1) & (tree.children_right == -1)
    leaf_samples = tree.n_node_samples[is_leaf]
    leaf_impurities = tree.impurity[is_leaf]
    return np.sum(leaf_samples * leaf_impurities) / np.sum(leaf_samples)

for d in depths:
    clf = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_errors.append(1 - clf.score(X_train, y_train))
    test_errors.append(1 - clf.score(X_test, y_test))
    weighted_leaf_entropies.append(get_tree_weighted_entropy(clf))

# Plot 1: Continuous Entropy Drop vs. Training & Test Error
fig, ax1 = plt.subplots(figsize=(11, 6))

ax1.plot(depths, train_errors, marker='o', color='blue', lw=2, label='Training Error (1 - Train Acc)')
ax1.plot(depths, test_errors, marker='s', color='red', lw=2, label='Test Error (1 - Test Acc)')
ax1.set_xlabel('Tree Max Depth (Model Complexity)')
ax1.set_ylabel('Classification Error Rate', color='black')
ax1.grid(True, linestyle='--', alpha=0.5)

# Secondary axis showing weighted leaf entropy continuously dropping toward 0
ax2 = ax1.twinx()
ax2.plot(depths, weighted_leaf_entropies, marker='^', color='green', linestyle=':', lw=2, label='Weighted Leaf Entropy (bits)')
ax2.set_ylabel('Weighted Leaf Entropy (bits)', color='green')

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.title('Overfitting Analysis: Continuous Entropy Reduction vs. Generalization Error')
plt.show()

# Identify Optimal Pre-Pruned Depth (Lowest Test Error)
optimal_depth = depths[np.argmin(test_errors)]
print(f"Optimal Pre-Pruned Max Depth: {optimal_depth}")
print(f"At Depth {optimal_depth} -> Train Error: {train_errors[optimal_depth-1]:.4f}, Test Error: {test_errors[optimal_depth-1]:.4f}, Leaf Entropy: {weighted_leaf_entropies[optimal_depth-1]:.4f}")

# 5-Fold Stratified Cross-Validation
best_dt = DecisionTreeClassifier(criterion='entropy', max_depth=optimal_depth, random_state=42)
cv_scores = cross_val_score(best_dt, X, y, cv=5)
print(f"5-Fold Stratified Cross-Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Bonus: Post-Pruning Demonstration via Cost-Complexity Pruning
full_tree = DecisionTreeClassifier(criterion='entropy', random_state=42).fit(X_train, y_train)
post_pruned_tree = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.015, random_state=42).fit(X_train, y_train)
print(f"\nFull Unpruned Tree Nodes: {full_tree.tree_.node_count} | Test Acc: {full_tree.score(X_test, y_test):.4f}")
print(f"Post-Pruned Tree Nodes:   {post_pruned_tree.tree_.node_count} | Test Acc: {post_pruned_tree.score(X_test, y_test):.4f}")

This code evaluates the bias-variance tradeoff by tracking how increasing Decision Tree complexity affects leaf entropy, overfitting, and generalization error. First, it loops through tree depths from 1 to 12, recording training error, test error, and sample-weighted leaf entropy via get_tree_weighted_entropy, then plots all three metrics on a dual-axis chart to prove that pushing leaf entropy to 0.000 drives training error to zero while eventually worsening test error (overfitting). Next, it selects the optimal pre-pruned max_depth that minimizes test error and validates its stability using 5-fold stratified cross-validation. Finally, it compares a full unpruned tree against a cost-complexity post-pruned tree (ccp_alpha=0.015), demonstrating how trimming weak bottom branches dramatically reduces node count while preserving strong test accuracy.

In [ ]:
# --- 9. MEASURING PERFORMANCE ON UNSEEN TEST DATA ---
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

# Fit optimal pruned model on training set and predict on unseen test set
best_dt.fit(X_train, y_train)
y_pred = best_dt.predict(X_test)
y_prob = best_dt.predict_proba(X_test)[:, 1] # Probability of 'Yes' (High Calorie Burn)
y_test_bin = (y_test == 'Yes').astype(int)

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['Yes', 'No'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Yes (High Cal)', 'No (Low Cal)'])
disp.plot(cmap='Blues')
plt.title(f"Confusion Matrix (Optimal Pruned Tree, Depth={optimal_depth})")
plt.show()

# Extract TP, FN, FP, TN manually
tp, fn, fp, tn = cm.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)           # TPR / Sensitivity
specificity = tn / (tn + fp)      # TNR
fpr_val = fp / (tn + fp)          # False Positive Rate (1 - Specificity)
f1 = 2 * (precision * recall) / (precision + recall)

print("=== MANUAL PERFORMANCE METRICS=====")
print(f"TP: {tp} | FN: {fn} | FP: {fp} | TN: {tn}")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall(TPR): {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"FPR:         {fpr_val:.4f}")
print(f"F1-Score:    {f1:.4f}\n")

print("=== SCIKIT-LEARN CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=['No (Low Cal)', 'Yes (High Cal)']))

# 2. ROC Curve & Precision-Recall Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left Plot: ROC Curve (TPR vs. FPR)
fpr, tpr, _ = roc_curve(y_test_bin, y_prob)
roc_auc = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'Pruned Decision Tree (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random Guessing (AUC = 0.500)')
axes[0].scatter([fpr_val], [recall], color='red', zorder=5, label=f'Operating Point ({fpr_val:.2f}, {recall:.2f})')
axes[0].set_xlabel('False Positive Rate (FPR = 1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (TPR / Recall)')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.5)

# Right Plot: Precision-Recall Curve
prec_curve, rec_curve, _ = precision_recall_curve(y_test_bin, y_prob)
ap_score = average_precision_score(y_test_bin, y_prob)
axes[1].plot(rec_curve, prec_curve, color='green', lw=2.5, label=f'Pruned Decision Tree (AP = {ap_score:.3f})')
axes[1].axhline(y=y_test_bin.mean(), color='gray', linestyle='--', label=f'Baseline Skew ({y_test_bin.mean():.2f})')
axes[1].set_xlabel('Recall (TPR)')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.5)

plt.tight_layout()
plt.show()

This code evaluates the predictive performance of your optimal pruned Decision Tree (best_dt) on the unseen test set using multiple classification metrics. First, it generates class predictions and predicted probabilities for the 'Yes' class, plotting a $2 \times 2$ Confusion Matrix to unpack True Positives (TP), False Negatives (FN), False Positives (FP), and True Negatives (TN). Using those counts, it manually computes key scalar evaluation metrics—Accuracy, Precision, Recall (TPR), Specificity (TNR), False Positive Rate (FPR), and F1-Score—alongside scikit-learn’s full classification report. Finally, it generates side-by-side ROC (plotting TPR vs. FPR with AUC and the model's exact operating point) and Precision-Recall curves (with Average Precision and a baseline skew reference line) to visualize classification trade-offs across probability thresholds.

Professor said We don't have to do rule based, so this is more like optional

In [ ]:
# --- 10. RULE-BASED CLASSIFIER: INDIRECT EXTRACTION, COVERAGE & ACCURACY) ---
from sklearn.tree import export_text

# 1. Indirect Method: Extract "if...then..." rules from our pruned Decision Tree
tree_rules = export_text(best_dt, feature_names=list(X.columns))
print("=== EXTRACTED DECISION TREE RULES (Indirect Method) ===")
print(tree_rules)

# 2. Evaluate Coverage and Accuracy on specific rules
# Combine X_train and y_train to test rule quality on training records
train_eval_df = X_train.copy()
train_eval_df['High_Calorie_Burn'] = y_train.values
n_total = len(train_eval_df)

def evaluate_rule(df, condition_mask, predicted_class, rule_name):
    covered = df[condition_mask]
    coverage = len(covered) / len(df)
    accuracy = (covered['High_Calorie_Burn'] == predicted_class).mean() if len(covered) > 0 else 0.0
    print(f"{rule_name}")
    print(f"   Coverage: {len(covered)}/{len(df)} ({coverage:.2%}) | Rule Accuracy: {accuracy:.2%}\n")

print("=== RULE COVERAGE & ACCURACY EVALUATION ===")
# Rule 1: High Session Duration -> High Calorie Burn ('Yes')
r1_mask = train_eval_df['Session_Duration (hours)'] > 1.25
evaluate_rule(train_eval_df, r1_mask, 'Yes', "R1: (Session_Duration > 1.25) -> High_Calorie_Burn = Yes")

# Rule 2: Conjunction of Duration AND Avg_BPM -> High Calorie Burn ('Yes')
r2_mask = (train_eval_df['Session_Duration (hours)'] > 1.15) & (train_eval_df['Avg_BPM'] > 145)
evaluate_rule(train_eval_df, r2_mask, 'Yes', "R2: (Session_Duration > 1.15) ^ (Avg_BPM > 145) -> High_Calorie_Burn = Yes")

# Rule 3: Short Session Duration -> Low Calorie Burn ('No')
r3_mask = train_eval_df['Session_Duration (hours)'] <= 1.10
evaluate_rule(train_eval_df, r3_mask, 'No', "R3: (Session_Duration <= 1.10) -> High_Calorie_Burn = No")

This code demonstrates Rule-Based Classification by translating tree splits into human-readable rules and measuring their statistical quality. First, it uses an indirect rule extraction method (export_text) to convert the branches of your trained Decision Tree (best_dt) into explicit if...then... decision rules. Next, it defines an evaluate_rule helper function that filters the training data using boolean masks (r1_mask, r2_mask, r3_mask) to test three specific antecedent conditions. For each rule, it calculates and prints both Coverage (the fraction of total records that satisfy the rule's condition) and Rule Accuracy (the fraction of those covered records that actually belong to the predicted class).

In [ ]:
# --- 11. DIRECT RULE LEARNING: FOIL INFORMATION GAIN & SEQUENTIAL COVERING  ---

def foil_gain(p0, n0, p1, n1):
    """Calculates FOIL Information Gain."""
    if p1 == 0:
        return 0.0
    return p1 * (np.log2(p1 / (p1 + n1)) - np.log2(p0 / (p0 + n0)))

# Initial empty rule R0: {} -> Yes
p0 = (train_eval_df['High_Calorie_Burn'] == 'Yes').sum()
n0 = (train_eval_df['High_Calorie_Burn'] == 'No').sum()
print(f"Initial Rule R0: {{}} -> Yes | p0 = {p0}, n0 = {n0}, Base Accuracy = {p0/(p0+n0):.2%}\n")

# Test candidate conjuncts to add to R0
candidate_conjuncts = {
    'Session_Duration > 1.25': train_eval_df['Session_Duration (hours)'] > 1.25,
    'Avg_BPM > 145':           train_eval_df['Avg_BPM'] > 145,
    'Fat_Percentage <= 22':    train_eval_df['Fat_Percentage'] <= 22,
    'Experience_Level == 3':   train_eval_df['Experience_Level'] == 3,
    'Age <= 35':               train_eval_df['Age'] <= 35
}

foil_results = []
for name, mask in candidate_conjuncts.items():
    sub = train_eval_df[mask]
    p1 = (sub['High_Calorie_Burn'] == 'Yes').sum()
    n1 = (sub['High_Calorie_Burn'] == 'No').sum()
    gain = foil_gain(p0, n0, p1, n1)
    acc = p1 / (p1 + n1) if (p1 + n1) > 0 else 0
    foil_results.append({'Candidate_Conjunct': name, 'p1': p1, 'n1': n1, 'Rule_Accuracy': round(acc, 4), 'FOIL_Gain': round(gain, 4)})

display(pd.DataFrame(foil_results).sort_values(by='FOIL_Gain', ascending=False))

# Simple Ordered Decision List
def predict_ordered_rules(df):
    preds = []
    for _, row in df.iterrows():
        # Rule 1 (Highest priority)
        if row['Session_Duration (hours)'] > 1.30:
            preds.append('Yes')
        # Rule 2
        elif (row['Session_Duration (hours)'] > 1.15) and (row['Avg_BPM'] > 142):
            preds.append('Yes')
        # Default Rule: {} -> No
        else:
            preds.append('No')
    return np.array(preds)

rule_test_preds = predict_ordered_rules(X_test)
print(f"Ordered Rule Set Test Accuracy: {(rule_test_preds == y_test).mean():.4f}")

In [ ]:
# --- 12. k-NEAREST NEIGHBORS (kNN) CLASSIFIER ---
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 1. Normalization (Z-Score Standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Compare Unscaled vs. Scaled kNN (k=5) to prove why normalization is required
knn_unscaled = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
knn_scaled = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
print(f"kNN (k=5) Test Accuracy WITHOUT Normalization: {knn_unscaled.score(X_test, y_test):.4f}")
print(f"kNN (k=5) Test Accuracy WITH Normalization:    {knn_scaled.score(X_test_scaled, y_test):.4f}\n")

# 2. Complexity Control: Tuning odd values of k
k_values = range(1, 26, 2) # Odd numbers: 1, 3, 5, ..., 25
knn_train_err = []
knn_test_err = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_train_scaled, y_train)
    knn_train_err.append(1 - knn.score(X_train_scaled, y_train))
    knn_test_err.append(1 - knn.score(X_test_scaled, y_test))

plt.figure(figsize=(10, 5))
plt.plot(k_values, knn_train_err, marker='o', color='blue', label='Training Error (k=1 -> 0.00 Overfit)')
plt.plot(k_values, knn_test_err, marker='s', color='red', label='Test Error')
plt.xlabel('Number of Neighbors (k) [Small k = Complex, Large k = Smooth]')
plt.ylabel('Classification Error Rate')
plt.title('kNN Bias-Variance Tradeoff: Impact of k')
plt.xticks(k_values)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

best_k = list(k_values)[np.argmin(knn_test_err)]
print(f"Optimal k (All Features): {best_k} | Test Accuracy: {1 - min(knn_test_err):.4f}")

# 3. Mitigating Irrelevant Features: Feature Selection + Distance Weighting
top_corr_cols = ['Session_Duration (hours)', 'Avg_BPM', 'Fat_Percentage', 'Experience_Level']
X_train_top = scaler.fit_transform(X_train[top_corr_cols])
X_test_top = scaler.transform(X_test[top_corr_cols])

knn_selected = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
knn_selected.fit(X_train_top, y_train)
print(f"kNN (k={best_k}, Distance-Weighted, Top 4 Correlated Features Only) Test Accuracy: {knn_selected.score(X_test_top, y_test):.4f}")

This code demonstrates direct rule learning by growing classification rules straight from the training data rather than extracting them from a decision tree. First, it defines the foil_gain function to calculate FOIL Information Gain ($p_1 \times [\log_2(\frac{p_1}{p_1+n_1}) - \log_2(\frac{p_0}{p_0+n_0})]$), starting from an empty rule $R_0: \{\} \rightarrow \text{Yes}$ and evaluating five candidate conditions to see which conjunct increases positive class purity the most. It then ranks those candidate conditions in a table by their FOIL Gain alongside their positive/negative instance counts ($p_1, n_1$) and rule accuracy. Finally, predict_ordered_rules implements a rank-ordered decision list—checking high-priority rules for the positive class ('Yes') sequentially and falling back on an empty default rule ({} -> No) when no condition triggers—before evaluating its overall accuracy on the unseen test set.

In [ ]:
# --- 13. MODEL COMPARISON SUMMARY (Eager vs. Lazy Learners) ---
comparison_df = pd.DataFrame([
    {'Model': f'Pruned Decision Tree (Depth={optimal_depth})', 'Learner Type': 'Eager (Global)', 'Test Accuracy': round(best_dt.score(X_test, y_test), 4)},
    {'Model': 'Ordered Rule-Based Classifier', 'Learner Type': 'Eager (Global)', 'Test Accuracy': round((rule_test_preds == y_test).mean(), 4)},
    {'Model': f'kNN Normalized (k={best_k}, All Features)', 'Learner Type': 'Lazy (Instance)', 'Test Accuracy': round(1 - min(knn_test_err), 4)},
    {'Model': f'kNN Weighted (k={best_k}, Top 4 Features)', 'Learner Type': 'Lazy (Instance)', 'Test Accuracy': round(knn_selected.score(X_test_top, y_test), 4)}
])
display(comparison_df)

In [ ]:
# --- 14. PROBABILITY & BAYES' THEOREM TUTORIAL ---
# Define Events on our dataset (df_tree)
# Event Y: High_Calorie_Burn == 'Yes'
# Event E: Session_Duration (hours) > 1.25
Y_mask = (df_tree['High_Calorie_Burn'] == 'Yes')
E_mask = (df_tree['Session_Duration (hours)'] > 1.25)

# 1. Prior Probabilities P(Y) and P(~Y)
p_Y = Y_mask.mean()
p_not_Y = 1 - p_Y  # Complement rule: Pr(E_bar) = 1 - Pr(E)

# 2. Likelihoods P(E | Y) and P(E | ~Y)
p_E_given_Y = E_mask[Y_mask].mean()
p_E_given_not_Y = E_mask[~Y_mask].mean()

# 3. Total Probability of Evidence P(E)
# P(E) = P(E|Y)*P(Y) + P(E|~Y)*P(~Y)
p_E = (p_E_given_Y * p_Y) + (p_E_given_not_Y * p_not_Y)

# 4. Posterior Probability via Bayes' Theorem P(Y | E)
p_Y_given_E = (p_E_given_Y * p_Y) / p_E

print("=== BAYES' THEOREM CALCULATION ON GYM DATA ===")
print(f"Prior P(High_Calorie = Yes):                 {p_Y:.4f}")
print(f"Likelihood P(Duration > 1.25 | Yes):         {p_E_given_Y:.4f}")
print(f"Likelihood P(Duration > 1.25 | No):          {p_E_given_not_Y:.4f}")
print(f"Evidence P(Duration > 1.25):                 {p_E:.4f}")
print(f"Posterior P(High_Calorie = Yes | Dur > 1.25): {p_Y_given_E:.4f} (Updated from {p_Y:.4f}!)")

This code demonstrates empirical probability and Bayes' Theorem on your gym dataset by calculating how observing new evidence updates the likelihood of burning high calories. First, it defines two binary events using boolean masks—target event $Y$ (High_Calorie_Burn == 'Yes') and evidence event $E$ (Session_Duration > 1.25)—and computes the baseline prior probability $P(Y)$ along with its complement $P(\neg Y) = 1 - P(Y)$. Next, it calculates the class-conditional likelihoods $P(E \mid Y)$ and $P(E \mid \neg Y)$ by measuring how often sessions exceed 1.25 hours within each calorie group. Finally, it applies the Law of Total Probability to find the overall probability of the evidence $P(E)$ and plugs all components into Bayes' Rule ($P(Y \mid E) = \frac{P(E \mid Y)P(Y)}{P(E)}$) to output the updated posterior probability of high calorie burn once a long workout session is observed.   

In [ ]:
# --- 15. NAÏVE BAYES CLASSIFIER: GAUSSIAN & MIXED DATA TYPES ---
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, auc

# Separate continuous features vs. categorical/ordinal features
cont_cols = [
    'Session_Duration (hours)', 'Avg_BPM', 'Max_BPM', 'Resting_BPM',
    'Fat_Percentage', 'Age', 'Weight (lb)', 'Height (in)', 'BMI',
    'Water_Intake (liters)', 'Workout_Frequency (days/week)'
]
cat_cols = ['Gender', 'Workout_Type', 'Experience_Level']

# 1. Train/Test Split using the exact same random_state=42 for fair comparison
X_cont_train, X_cont_test, y_nb_train, y_nb_test = train_test_split(
    df_tree[cont_cols], df_tree['High_Calorie_Burn'], test_size=0.2, random_state=42, stratify=df_tree['High_Calorie_Burn']
)
X_cat_train, X_cat_test = train_test_split(
    df_tree[cat_cols], test_size=0.2, random_state=42, stratify=df_tree['High_Calorie_Burn']
)[0:2]

# Encode categorical columns into integer indices for CategoricalNB
encoder = OrdinalEncoder()
X_cat_train_enc = encoder.fit_transform(X_cat_train)
X_cat_test_enc = encoder.transform(X_cat_test)

# 2. Model A: Gaussian Naïve Bayes (Continuous features only)
gnb = GaussianNB()
gnb.fit(X_cont_train, y_nb_train)
gnb_acc = gnb.score(X_cont_test, y_nb_test)

# 3. Model B: Combined GaussianNB + CategoricalNB with Laplace Smoothing
# Note: CategoricalNB uses alpha=1.0 (Laplace estimate) by default
cnb = CategoricalNB(alpha=1.0)
cnb.fit(X_cat_train_enc, y_nb_train)

# Combine joint log-probabilities: log P(X_cont, X_cat | Y) + log P(Y)
log_prob_joint_test = (
    gnb.predict_joint_log_proba(X_cont_test) +
    cnb.predict_joint_log_proba(X_cat_test_enc) -
    np.log(gnb.class_prior_)
)

# Convert log-probabilities to normalized posterior probabilities via softmax
log_prob_norm = log_prob_joint_test - np.max(log_prob_joint_test, axis=1, keepdims=True)
probs_mixed = np.exp(log_prob_norm) / np.sum(np.exp(log_prob_norm), axis=1, keepdims=True)

mixed_preds = gnb.classes_[np.argmax(log_prob_joint_test, axis=1)]
mixed_acc = (mixed_preds == y_nb_test).mean()

print(f"GaussianNB (Continuous Features Only) Test Accuracy:      {gnb_acc:.4f}")
print(f"Mixed Naïve Bayes (GaussianNB + CategoricalNB) Accuracy:  {mixed_acc:.4f}\n")

# 4. Evaluate Mixed Naïve Bayes
print("=== MIXED NAÏVE BAYES CLASSIFICATION REPORT ===")
print(classification_report(y_nb_test, mixed_preds))

cm_nb = confusion_matrix(y_nb_test, mixed_preds, labels=['Yes', 'No'])
ConfusionMatrixDisplay(confusion_matrix=cm_nb, display_labels=['Yes (High Cal)', 'No (Low Cal)']).plot(cmap='Purples')
plt.title("Confusion Matrix: Mixed Naïve Bayes Classifier")
plt.show()

This code implements a Naïve Bayes Classifier tailored to handle mixed data types by separating continuous attributes from categorical/ordinal attributes and applying the appropriate probability distribution to each. First, it splits the dataset using the exact same random_state=42 and stratified sampling for a fair model comparison, encoding categorical features (Gender, Workout_Type, Experience_Level) into integer indices via OrdinalEncoder. Next, it fits a baseline GaussianNB on continuous features alone (assuming normal distributions) and a CategoricalNB(alpha=1.0) on categorical features (applying Laplace smoothing to prevent zero-probability issues). It then combines both models by summing their joint log-probabilities and subtracting $\log P(Y)$ so the class prior isn't counted twice, converts the resulting scores into normalized posterior probabilities via softmax, and evaluates the combined model's accuracy, classification report, and confusion matrix on the test set.

In [ ]:
# --- 16. UNIFIED MODEL COMPARISON & ROC CURVE OVERLAY ---
y_test_bin = (y_test == 'Yes').astype(int)

# Get predicted probabilities for class 'Yes' across all three model families
dt_probs = best_dt.predict_proba(X_test)[:, 1]
knn_probs = knn_selected.predict_proba(X_test_top)[:, 1]
nb_probs = probs_mixed[:, list(gnb.classes_).index('Yes')]

plt.figure(figsize=(9, 6))
for name, probs, color in [
    (f'Decision Tree (Depth={optimal_depth})', dt_probs, 'darkorange'),
    (f'kNN (k={best_k}, Weighted)', knn_probs, 'blue'),
    ('Mixed Naïve Bayes (Gaussian + Categorical)', nb_probs, 'purple')
]:
    fpr, tpr, _ = roc_curve(y_test_bin, probs)
    model_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2.2, color=color, label=f'{name} (AUC = {model_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.500)')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR / Recall)')
plt.title('ROC Curve Comparison: Decision Tree vs. kNN vs. Naïve Bayes')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.5)
plt.show()

final_summary = pd.DataFrame([
    {'Model': f'Decision Tree (Entropy, Depth={optimal_depth})', 'Paradigm': 'Tree / Eager', 'Test Accuracy': round(best_dt.score(X_test, y_test), 4), 'ROC AUC': round(auc(*roc_curve(y_test_bin, dt_probs)[:2]), 4)},
    {'Model': 'Ordered Rule-Based Classifier', 'Paradigm': 'Rules / Eager', 'Test Accuracy': round((rule_test_preds == y_test).mean(), 4), 'ROC AUC': 'N/A (Hard Rules)'},
    {'Model': f'kNN (k={best_k}, Normalized, Top 4 Features)', 'Paradigm': 'Instance / Lazy', 'Test Accuracy': round(knn_selected.score(X_test_top, y_test), 4), 'ROC AUC': round(auc(*roc_curve(y_test_bin, knn_probs)[:2]), 4)},
    {'Model': 'Mixed Naïve Bayes (Gaussian + Categorical)', 'Paradigm': 'Probabilistic / Eager', 'Test Accuracy': round(mixed_acc, 4), 'ROC AUC': round(auc(*roc_curve(y_test_bin, nb_probs)[:2]), 4)}
])
display(final_summary)

This code benchmarks and visually compares the classification performance of all four supervised model paradigms side-by-side on the unseen test set. First, it converts the ground-truth labels (y_test) into binary 0/1 values and extracts the predicted probabilities for the 'Yes' class from your Pruned Decision Tree, Distance-Weighted kNN, and Mixed Naïve Bayes models. Next, it plots all three ROC Curves on a single graph against a diagonal random-guessing baseline (AUC = 0.500), computing the Area Under the Curve (AUC) for each model to show how effectively each classifier separates high-calorie from low-calorie workouts across all probability thresholds. Finally, it compiles a master final_summary DataFrame listing each model’s learning paradigm (eager vs. lazy), test accuracy, and ROC AUC score (marking ROC AUC as 'N/A' for the hard-threshold Ordered Rule Classifier since it outputs discrete class labels rather than continuous probability scores).

Professor did say it is ok not to include Association Analysis.

In [ ]:
# --- 17. ASSOCIATION ANALYSIS: APRIORI & INTERESTINGNESS METRICS ---
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Select focused subset of features
assoc_df = pd.DataFrame()

# Keep natural categorical/ordinal attributes
assoc_df['Workout'] = 'Workout=' + df_clean['Workout_Type'].astype(str)
exp_map = {1: 'Exp=Beginner', 2: 'Exp=Intermediate', 3: 'Exp=Expert'}
assoc_df['Experience'] = df_clean['Experience_Level'].map(exp_map)

# Discretize key continuous attributes into interpretable binary tiers (Median split)
assoc_df['Duration'] = np.where(df_clean['Session_Duration (hours)'] > df_clean['Session_Duration (hours)'].median(),
                                'Duration=Long', 'Duration=Short')
assoc_df['HeartRate'] = np.where(df_clean['Avg_BPM'] > df_clean['Avg_BPM'].median(),
                                 'AvgBPM=High', 'AvgBPM=Moderate')
assoc_df['BodyFat'] = np.where(df_clean['Fat_Percentage'] <= df_clean['Fat_Percentage'].median(),
                               'BodyFat=Lower', 'BodyFat=Higher')
assoc_df['Calories'] = np.where(df_clean['Calories_Burned'] > df_clean['Calories_Burned'].median(),
                                'Calories=High', 'Calories=Low')

# 2. One-Hot Encode into Market-Basket Boolean Matrix (N transactions x d items)
basket = pd.get_dummies(assoc_df, prefix='', prefix_sep='').astype(bool)
print(f"Transaction Matrix Shape: {basket.shape[0]} gym sessions (N) x {basket.shape[1]} binary items (d)\n")

# 3. Step 1 of Apriori: Frequent Itemset Generation
# Using minsup = 0.15 (itemset must appear in at least 15% of sessions)
frequent_itemsets = apriori(basket, min_support=0.15, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
print(f"Total Frequent Itemsets Found (minsup >= 0.15): {len(frequent_itemsets)}")
display(frequent_itemsets.sort_values(by='support', ascending=False).head(10))

# 4. Step 2 of Apriori: Rule Generation
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.60)

# 5. Compute Objective Interestingness Measures from Contingency Table
# For rule X -> Y:
# f11 = N * s(X U Y)
# f1+ = N * s(X)
# f+1 = N * s(Y)
N = len(basket)
f11 = rules['support'] * N
f1_plus = rules['antecedent support'] * N
f_plus1 = rules['consequent support'] * N
f10 = f1_plus - f11
f01 = f_plus1 - f11
f00 = N - (f11 + f10 + f01)

# Interest Factor / Lift: I(X,Y) = s(X,Y) / (s(X)*s(Y))
rules['Interest_Factor (Lift)'] = rules['lift']

# Interest Score / Cosine Similarity: IS(X,Y) = s(X,Y) / sqrt(s(X)*s(Y))
rules['Interest_Score (Cosine)'] = rules['support'] / np.sqrt(rules['antecedent support'] * rules['consequent support'])

# Odds Ratio: OR(X,Y) = (f11 * f00) / (f10 * f01)
rules['Odds_Ratio'] = (f11 * f00) / (f10 * f01 + 1e-9)

# Clean formatting for display
display_cols = [
    'antecedents', 'consequents', 'support', 'confidence',
    'Interest_Factor (Lift)', 'Interest_Score (Cosine)', 'Odds_Ratio'
]
rules_clean = rules[display_cols].copy()
rules_clean['antecedents'] = rules_clean['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_clean['consequents'] = rules_clean['consequents'].apply(lambda x: ', '.join(list(x)))

# Filter for positively associated rules (Interest Factor > 1.2) and sort by Interest Score (Cosine)
top_rules = rules_clean[rules_clean['Interest_Factor (Lift)'] > 1.2].sort_values(
    by='Interest_Score (Cosine)', ascending=False
).round(3)

print("\n=== TOP NON-TRIVIAL ASSOCIATION RULES (Filtered by Lift > 1.2, Ranked by Cosine IS) ===")
display(top_rules.head(12))

This code performs unsupervised association rule mining using the Apriori algorithm and evaluates discovered patterns using objective interestingness metrics. First, it transforms six key gym attributes into categorical labels—mapping Experience_Level and applying median splits to continuous features (Duration, HeartRate, BodyFat, and Calories)—before one-hot encoding them into a boolean market-basket transaction matrix (basket). Next, it executes the two-stage Apriori pipeline: mining all itemsets that appear in at least 15% of sessions (min_support=0.15) and generating association rules with at least 60% confidence (min_threshold=0.60). Finally, it reconstructs the underlying $2 \times 2$ contingency table counts ($f_{11}, f_{10}, f_{01}, f_{00}$) for every rule to compute Interest Factor (Lift), Interest Score (Cosine Similarity), and Odds Ratio, filtering for positively associated rules (Lift > 1.2) and ranking the strongest, non-trivial workout co-occurrences by Cosine Similarity.   

In [ ]:
# --- 18. CONTINGENCY TABLE VERIFICATION  ---
# Example Rule: {Duration=Long} -> {Calories=High}
X = basket['Duration=Long']
Y = basket['Calories=High']

ct = pd.crosstab(
    pd.Series(np.where(X, 'Duration=Long', '~Duration=Long'), name='Antecedent (X)'),
    pd.Series(np.where(Y, 'Calories=High', '~Calories=High'), name='Consequent (Y)'),
    margins=True
)
print("=== 2x2 CONTINGENCY TABLE: {Duration=Long} -> {Calories=High} ===")
display(ct)

# Extract cell counts f11, f10, f01, f00
f_11 = ((X) & (Y)).sum()
f_10 = ((X) & (~Y)).sum()
f_01 = ((~X) & (Y)).sum()
f_00 = ((~X) & (~Y)).sum()

sup = f_11 / N
conf = f_11 / (f_11 + f_10)
lift = (N * f_11) / ((f_11 + f_10) * (f_11 + f_01))
cosine_is = f_11 / np.sqrt((f_11 + f_10) * (f_11 + f_01))
odds_ratio = (f_11 * f_00) / (f_10 * f_01)

print(f"Support:                 {sup:.4f}")
print(f"Confidence:              {conf:.4f}")
print(f"Interest Factor (Lift):  {lift:.4f} (> 1 confirms positive association)")
print(f"Interest Score (Cosine): {cosine_is:.4f} (balances association strength & support)")
print(f"Odds Ratio:              {odds_ratio:.4f}")

This final code block manually verifies the objective interestingness metrics for a single association rule ({Duration=Long} -> {Calories=High}) using a $2 \times 2$ contingency table. First, it isolates the boolean columns for the antecedent (Duration=Long) and consequent (Calories=High) from the market-basket matrix and displays a cross-tabulated frequency table with row and column totals (margins=True). Next, it extracts the four exact cell counts—$f_{11}$ (both true), $f_{10}$ (only $X$ true), $f_{01}$ (only $Y$ true), and $f_{00}$ (neither true)—directly from the boolean masks. Finally, it plugs those cell frequencies into the standard contingency formulas to compute and print Support, Confidence, Interest Factor (Lift), Interest Score (Cosine Similarity), and Odds Ratio, confirming the mathematical accuracy of the mlxtend output from the previous cell.